In [10]:
import yfinance as yf 
from bcb import sgs # API do BC 
import pandas as pd
from functools import reduce
from datetime import timedelta, datetime

# src/data_loader.py

def extract_yf(ticker_datas: dict, variacao=True) -> pd.DataFrame: #vai receber um dicionário com os tickers e as datas de início e fim
    frames = []
    for ticker, (start, end) in ticker_datas.items(): # loop por cada ativo e suas datas
        df = yf.download(ticker, start=start, end=end) # baixar os dados do yahoo finance
        if df.empty: # pular ativos sem dados
            print(f"Nenhum dado encontrado para {ticker}")
            continue
        df = df[["Close"]].copy() # trabalhar com o preço de fechamento
        df.rename(columns={"Close": f"{ticker}_preco"}, inplace=True) # renomear a coluna para o ticker_preço
        if variacao: # calcular a variação percentual se solicitado (true)
            df[f"{ticker}_var_pct"] = df[f"{ticker}_preco"].pct_change() * 100
        df["Ticker"] = ticker
        df.reset_index(inplace=True)
        frames.append(df) #adicionar o DataFrame à lista de frames

    result = reduce(lambda left, right: pd.merge(left, right, on="Date", how="outer"), frames)
    return result

def extract_bcb(series: dict, start: str, end: str) -> pd.DataFrame: # API só aceita 10 anos de dados por vez

    start_date = pd.to_datetime(start)
    end_date = pd.to_datetime(end)
    max_range = timedelta(days=365 * 10 - 1)  # 10 anos menos 1 dia

    dfs = []

    while start_date < end_date:
        next_end = min(start_date + max_range, end_date)
        df_parcial = sgs.get(series, start=start_date.strftime('%Y-%m-%d'), end=next_end.strftime('%Y-%m-%d'))
        dfs.append(df_parcial)
        start_date = next_end + timedelta(days=1)

    df_final = pd.concat(dfs)
    df_final.reset_index(inplace=True)
    return df_final




In [ ]:
#exemplo de aplicação 

# Yfinance 

ticker_datas = {
    "CL=F": ("2004-01-01", "2023-10-31"), # Petróleo - ta diário, vamos mudar para mensal (acumulado no mês)
    "USDBRL=X": ("2005-01-01", "2023-10-31"), # Variação do Dólar - ta diário, vamos mudar para mensal (acumulado no mês)
}

# BC
series_bc = {
    "IPCA": 433,                 # Índice Nacional de Preços ao Consumidor Amplo - ao mês (ta a cada 3 meses - ajustar depois)
    "SELIC": 11,               # SELIC efetiva (% a.d.)
    "SELIC_META": 432,           # Meta SELIC (% a.a.) - mas ta registrada ao dia, tem que arrumar isso
    "IND_DESMP": 24369,          # Taxa de desocupação (É como o IBGE chama a taxa de desemprego) - PNADC
    "IPCA_ALIMENTOS": 1635,       # Variação da cesta básica (São Paulo) - não ta funcionando - mudei para IPCA de alimentos - ao mês
    "PETROLEO_VAR": 4769,        # Variação percentual do petróleo - ao mês
    "IC-Br Agropecuária": 27575,            # Variação percentual da soja - naõ ta funcionando - Mudei para IC-Br Agropecuária - ao mês
    "PIB": 7326,                 # PIB  a preços de mercado - ao ano
    "FED": 11710,                # Taxa de juros do FED (Federal Reserve) - ao mês
    "DIVIDA_EXTERNA": 2073   # Dívida externa do BC e Governo Federal - ao mês
}





start = "2004-01-01"
end = "2023-10-31"

df_yf = extract_yf(ticker_datas = ticker_datas, variacao=True)
df_bcb = extract_bcb(series_bc, start, end)




,Date,IPCA,SELIC,SELIC_META,IND_DESMP,IPCA_ALIMENTOS,PETROLEO_VAR,IC-Br Agropecuária,PIB,FED,DIVIDA_EXTERNA
0,2004-01-01,0.76,NaN,16.50,NaN,0.88,20.06,99.53,5.76,63255.0,154073.0
1,2004-01-02,NaN,0.060076,16.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2004-01-03,NaN,NaN,16.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2004-01-04,NaN,NaN,16.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2004-01-05,NaN,0.060076,16.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
7241,2023-10-27,NaN,0.047279,12.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7242,2023-10-28,NaN,NaN,12.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7243,2023-10-29,NaN,NaN,12.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7244,2023-10-30,NaN,0.047279,12.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
df_yf.to_csv("../data/raw/yf_data.csv", index=False)
df_bcb.to_csv("../data/raw/bcb_data.csv", index=False)